# AI Bootcamp — Section 4: **Real Local Multi-Agent System** (Ollama + Qwen3)

**आपका नया Notebook — बिना AWS, बिना Cloud API Key।**

यह उसी SwiftCart (fictional online retailer) case को जारी रखता है जो आपने पहले Offline Section 4 में किया था। इस बार **News Agent**, **Financial Agent**, **Sentiment Agent**, और **Supervisor Agent** चारों स्थानीय Qwen3 मॉडल से वास्तविक text generation करते हैं।

> ⚠️ इस notebook में सभी उदाहरण काल्पनिक हैं। इसमें live news, real financial market data, AWS, web search, या paid APIs का उपयोग **नहीं** होता।

**Run कैसे करें:** PowerShell में JupyterLab खुला हो → notebook खोलें → हर **Code Cell** पर क्रम से `Shift + Enter`। `Run All` के बजाय शुरुआत में एक-एक cell चलाना बेहतर है। आपके लैपटॉप में Ollama तथा `qwen3:1.7b` पहले से चल चुके हैं।

**Architecture:** Question → तीन विशेषज्ञ Workers (क्रम से) → Supervisor → Final Report. इसे *deterministic sequential orchestration* कहते हैं। यह छोटे local models के लिए supervisor के automatic tool-calling से अधिक भरोसेमंद है।

## Cell 1 — Ollama Local Server और Qwen3 की जाँच

यह केवल `localhost:11434` से कनेक्शन जाँचता है। अगर `Connection refused` आए तो दूसरी PowerShell Window में `ollama serve` चलाएँ (यदि Ollama पहले से नहीं चल रहा)।

In [ ]:
# Cell 1 — Check Ollama server and installed model (no generation)
import json
from urllib.request import urlopen
from urllib.error import URLError, HTTPError

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "qwen3:1.7b"

try:
    with urlopen(f"{OLLAMA_URL}/api/tags", timeout=10) as response:
        tags = json.load(response)
    model_names = [m.get("name", "") for m in tags.get("models", [])]
    print("PASS: Ollama server is running locally")
    print("Installed models:", model_names)
    if MODEL not in model_names:
        print(f"MODEL NOT FOUND: In PowerShell, run: ollama pull {MODEL}")
    else:
        print(f"PASS: {MODEL} is available")
except (URLError, TimeoutError, OSError) as exc:
    print("ERROR: JupyterLab cannot reach the Ollama server")
    print("Try: open a separate PowerShell window and run: ollama serve")
    print("Technical error:", type(exc).__name__, str(exc)[:300])
    raise

## Cell 2 — SwiftCart के काल्पनिक Data और Python से सत्यापित Metrics

**Revenue Growth** = (Current - Previous) / Previous. **Operating Margin** = (Revenue - Costs) / Revenue. Calculations Python में की गई हैं ताकि model गलत arithmetic न बना दे।

In [ ]:
# Cell 2 — Fictional dataset reused from the earlier Section 4 notebook
CASE = {
    "company": "SwiftCart (fictional online retailer)",
    "finance": {
        "monthly_revenue_inr": 1_200_000,
        "monthly_cost_inr": 1_080_000,
        "prior_month_revenue_inr": 1_100_000,
    },
    "news": [
        "A rival introduced free same-day delivery (fictional observation)",
        "Retailers are emphasizing reliable delivery tracking (fictional observation)",
    ],
    "feedback": {
        "positive_percent": 62,
        "negative_percent": 23,
        "neutral_percent": 15,
        "top_complaint": "late delivery",
    },
}

f = CASE["finance"]
revenue_growth_pct = (f["monthly_revenue_inr"] - f["prior_month_revenue_inr"]) / f["prior_month_revenue_inr"] * 100
operating_profit_inr = f["monthly_revenue_inr"] - f["monthly_cost_inr"]
operating_margin_pct = operating_profit_inr / f["monthly_revenue_inr"] * 100

assert round(revenue_growth_pct, 2) == 9.09
assert operating_margin_pct == 10.0
print(f"PASS: Revenue growth = {revenue_growth_pct:.2f}%")
print(f"PASS: Operating margin = {operating_margin_pct:.1f}%")
print(f"Operating profit = INR {operating_profit_inr:,}")

## Cell 3 — Ollama से बात करने वाला Python Function

यह official Ollama `/api/chat` local endpoint उपयोग करता है। `think=False` से Qwen3 छोटा, सीधा answer देता है; `stream=False` के कारण एक response मिलता है। कोई Python package install नहीं करना।

In [ ]:
# Cell 3 — Standard-library only local Ollama client
from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError


def ollama_chat(system_prompt, user_prompt, max_tokens=180):
    body = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "stream": False,
        "think": False,
        "keep_alive": "10m",
        "options": {"temperature": 0.2, "num_predict": max_tokens, "num_ctx": 2048},
    }
    request = Request(
        f"{OLLAMA_URL}/api/chat",
        data=json.dumps(body).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urlopen(request, timeout=300) as response:
            result = json.load(response)
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")[:450]
        raise RuntimeError(f"Ollama returned HTTP {exc.code}: {detail}") from exc
    except (URLError, TimeoutError, OSError) as exc:
        raise RuntimeError(
            "Ollama connection failed. Confirm Ollama is running and "+
            "the model exists. Technical error: "+str(exc)[:250]
        ) from exc
    answer = result.get("message", {}).get("content", "").strip()
    if not answer:
        raise RuntimeError("Ollama produced an empty response; check the model and try again.")
    return answer

print("PASS: Local Ollama client configured; no AWS credentials needed")

## Cell 4 — Quick Test: Qwen3 से वास्तविक Local Answer

पहले केवल 1 छोटा model call। यह पूरा होते ही Model Connection साबित हो जाएगा। CPU पर कुछ समय लग सकता है।

In [ ]:
# Cell 4 — One free local AI call
reply = ollama_chat(
    "You are a concise Hindi teacher.",
    "Explain in one simple Hindi sentence what a supervisor agent does.",
    max_tokens=90,
)
print("PASS: Local Qwen3 answered:\n")
print(reply)

## Cell 5 — तीन Worker Agents + Supervisor Agent बनाएँ

इन चारों Agents के अलग-अलग **system prompts** हैं। हर Agent उसी local AI model का उपयोग करेगा, लेकिन अलग विशेषज्ञ भूमिका निभाएगा।

In [ ]:
# Cell 5 — Dedicated roles using the same real local model
class LocalAgent:
    def __init__(self, name, system_prompt):
        self.name = name
        self.system_prompt = system_prompt

    def run(self, task, max_tokens=180):
        print(f"\nRunning {self.name} on local Qwen3 ...")
        response = ollama_chat(self.system_prompt, task, max_tokens=max_tokens)
        print(f"PASS: {self.name} finished")
        return response

news_agent = LocalAgent(
    "news_agent",
    "You are a business news specialist. Use ONLY provided fictional observations. "
    "Do NOT say you browsed the web. In Hindi, explain up to 2 possible business implications. "
    "Separate facts and hypotheses. Keep it concise.",
)
financial_agent = LocalAgent(
    "financial_agent",
    "You are a financial data analyst. Use ONLY provided fictional numbers and "
    "the precomputed growth and margin. Do not recalculate or invent data. "
    "In Hindi, write 2 concise findings and 1 limitation.",
)
sentiment_agent = LocalAgent(
    "sentiment_agent",
    "You are a customer sentiment analyst. Use ONLY given fictional survey percentages. "
    "In Hindi, state the biggest customer issue and a sensible next step. "
    "Mention that the sample size is unknown.",
)
supervisor_agent = LocalAgent(
    "research_supervisor",
    "You are the supervisor of three expert AI workers. Use their reports to synthesize "
    "one short Hindi business report, with headings: Summary, Metrics, Priorities, Limitations. "
    "Do not invent facts and always label this a fictional training case. "
    "Use exact numeric metrics given in the worker reports.",
)
print("PASS: 3 local AI Workers and 1 local AI Supervisor ready")

## Cell 6 — Worker Agents चलाएँ (3 real local AI calls)

पहली बार एक-एक worker क्रम से Qwen3 से जवाब देगा। तीनों के Results `worker_reports` में Store होंगे।

In [ ]:
# Cell 6 — Sequential worker orchestration; no web and no paid requests
context = (
    f"FICTIONAL CASE: {CASE['company']}\n"
    f"Revenue INR {f['monthly_revenue_inr']:,}; costs INR {f['monthly_cost_inr']:,}; "
    f"prior revenue INR {f['prior_month_revenue_inr']:,}.\n"
    f"Precomputed revenue growth: {revenue_growth_pct:.2f}%; "
    f"operating profit INR {operating_profit_inr:,}; "
    f"operating margin: {operating_margin_pct:.1f}%.\n"
    f"Fictional news: {CASE['news']}.\n"
    f"Survey: positive {CASE['feedback']['positive_percent']}%, "
    f"negative {CASE['feedback']['negative_percent']}%, "
    f"neutral {CASE['feedback']['neutral_percent']}%. "
    f"Main complaint: {CASE['feedback']['top_complaint']}. "
    f"Sample size unknown."
)

worker_reports = {}
for worker in (news_agent, financial_agent, sentiment_agent):
    worker_reports[worker.name] = worker.run(context)
    print(worker_reports[worker.name][:1200])

assert set(worker_reports) == {"news_agent", "financial_agent", "sentiment_agent"}
assert all(worker_reports.values())
print("\nPASS: All 3 Worker Agents produced real Qwen3 responses")

## Cell 7 — Supervisor को Workers के Reports दें (1 real local AI call)

Supervisor तीनों outputs को पढ़कर अंतिम रिपोर्ट बनाता है। **यह Python-controlled delegation** है; छोटे Qwen3 मॉडल से भरोसेमंद multi-agent प्रदर्शन के लिए जानबूझकर fixed worker order रखा गया है।

In [ ]:
# Cell 7 — Supervisor synthesis, fourth local AI call
assert len(worker_reports) == 3, "Run Cell 6 first"
question = "What operational priorities should SwiftCart choose next quarter? Answer in Hindi."
combined = (
    f"QUESTION: {question}\n\n"
    f"PRECOMPUTED FACTS: revenue growth = {revenue_growth_pct:.2f}%; "
    f"operating margin = {operating_margin_pct:.1f}%; "
    f"operating profit = INR {operating_profit_inr:,}.\n\n"
    + "\n\n".join(f"[{agent}]\n{body}" for agent, body in worker_reports.items())
)
final_report = supervisor_agent.run(combined, max_tokens=320)
print("\n========== FINAL LOCAL MULTI-AGENT REPORT ==========\n")
print(final_report)
print("\nPASS: Supervisor synthesized the three real worker reports")

## Cell 8 — रिपोर्ट Save करें

Report आपके JupyterLab के current folder में `section4_ollama_supervisor_report.txt` नाम से बनेगी।

In [ ]:
# Cell 8 — Save plain-text artifact to current notebook directory
from pathlib import Path
assert isinstance(final_report, str) and final_report.strip(), "Run Cell 7 first"
output = Path("section4_ollama_supervisor_report.txt")
output.write_text(
    "AI Bootcamp Section 4 - Ollama Qwen3 Supervisor-Worker\n"
    "Fictional SwiftCart training case: NOT live market/news data.\n\n"
    + final_report + "\n",
    encoding="utf-8",
)
print("PASS: Section 4 local multi-agent workflow completed")
print("Saved report:", output.resolve())

## Troubleshooting / अगला कदम

- **Connection refused:** Ollama Desktop App खोलें; जरूरत हो तो नई PowerShell में `ollama serve`। अगर `address already in use` आए, Ollama server शायद पहले से चल रहा है; Cell 1 चलाकर देखें।
- **Model not found (404):** PowerShell में `ollama list` जाँचें, फिर जरूरत हो तो `ollama pull qwen3:1.7b`।
- **Slow:** Qwen3 local model CPU पर धीरे चल सकता है। Notebook में एक-एक Cell चलाएँ; Cells 6 और 7 के दौरान Kernel busy दिखना सामान्य है।
- **Memory / Out-of-memory:** अन्य heavy Apps बंद करें। जरूरत पड़ने पर MODEL को `qwen3:0.6b` करें (पहले `ollama pull qwen3:0.6b` चलाएँ)। छोटा मॉडल कम सटीक हो सकता है।
- **AWS error:** गलती से पुरानी `01_supervisor_worker_starter.ipynb` के AWS Cells न चलाएँ। इस नए Notebook में AWS इस्तेमाल ही नहीं होता।
- **Do not expose local Ollama port to the public internet.** Localhost (`127.0.0.1`) ही रखें।

**What counts as done:** Cell 1 model available → Cell 4 live local answer → Cell 6 three workers → Cell 7 Supervisor → Cell 8 report saved. यह Section 4 का **Supervisor–Worker Local variant** पूरा करता है; Section 4 के बाकी Agent-as-Tool, Graph और Swarm practicals अभी बाकी होंगे।